# ORDER BY, LIMIT, and OFFSET

## Overview
`ORDER BY` specifies the sequence of result rows. `LIMIT` caps how many rows are returned. `OFFSET` skips a number of leading rows. Together they are the foundation of pagination and top-N queries.

**Syntax:**
```sql
SELECT ...
FROM   ...
ORDER BY col1 [ASC|DESC] [NULLS FIRST|NULLS LAST],
         col2 [ASC|DESC]
LIMIT  n
OFFSET k;
```

**Key facts:**

| | SQLite | PostgreSQL |
|---|---|---|
| Default sort direction | ASC | ASC |
| NULL sort (ASC) | NULLs last | NULLs last |
| NULL sort (DESC) | NULLs first | NULLs first |
| Explicit control | `CASE WHEN col IS NULL` workaround | `NULLS FIRST` / `NULLS LAST` |
| SQL standard alternative to LIMIT | — | `FETCH FIRST n ROWS ONLY` |
| Keyset pagination | Supported | Supported (`WHERE id > last_id`) |

**Without ORDER BY, row order is undefined.** The database may return the same query in different orders across executions, especially after inserts, deletes, or vacuuming. Never rely on implicit order.

---

In [ ]:
import sqlite3
import pandas as pd

conn = sqlite3.connect(':memory:')
cur  = conn.cursor()

cur.executescript("""
CREATE TABLE prescriptions (
    rx_id        INTEGER PRIMARY KEY,
    patient_id   INTEGER,
    drug_name    TEXT,
    dose_mg      REAL,
    prescribed   TEXT,
    cost_cad     REAL,
    refills      INTEGER
);

INSERT INTO prescriptions VALUES
  (1,  1,  'Metformin',    500,  '2023-01-10', 28.50, 5),
  (2,  2,  'Atorvastatin',  20,  '2023-02-01', 45.00, 3),
  (3,  3,  'Lisinopril',    10,  '2023-02-15', 18.75, 5),
  (4,  1,  'Lisinopril',     5,  '2023-03-01', 12.25, 3),
  (5,  4,  'Metformin',   1000,  '2023-03-20', 52.00, 5),
  (6,  5,  'Atorvastatin',  40,  '2023-04-05', 67.50, 3),
  (7,  2,  'Metoprolol',    25,  '2023-04-18', 22.00, 5),
  (8,  6,  'Ramipril',       5,  '2023-05-02', 19.50, 3),
  (9,  7,  'Atorvastatin',  80,  '2023-05-15', 89.00, 1),
  (10, 3,  'Metformin',    500,  '2023-06-01', 28.50, 5),
  (11, NULL,'Pantoprazole', 20,  '2023-06-12', 34.00, 3),
  (12, 8,  'Metoprolol',    50,  '2023-07-01', 38.00, 5),
  (13, 9,  'Lisinopril',    10,  '2023-07-20', 18.75, 5),
  (14, 10, 'Ramipril',      10,  '2023-08-05', 24.00, 3);
""")
conn.commit()

print('prescriptions table ready — 14 rows')
print(pd.read_sql('SELECT * FROM prescriptions', conn).to_string(index=False))


---
## Single and multi-column ORDER BY


In [ ]:
# Single column, descending
print('=== Most expensive prescriptions first ===')
q = """
SELECT rx_id, drug_name, dose_mg, cost_cad
FROM   prescriptions
ORDER BY cost_cad DESC
"""
print(pd.read_sql(q, conn).to_string(index=False))

# Multi-column: primary sort then tiebreaker
print()
print('=== Drug name ASC, then cost DESC within each drug ===')
q2 = """
SELECT drug_name, dose_mg, cost_cad, patient_id
FROM   prescriptions
ORDER BY drug_name ASC, cost_cad DESC
"""
print(pd.read_sql(q2, conn).to_string(index=False))

# ORDER BY expression
print()
print('=== Sorted by cost per refill (computed expression in ORDER BY) ===')
q3 = """
SELECT drug_name, cost_cad, refills,
       ROUND(cost_cad / refills, 2) AS cost_per_refill
FROM   prescriptions
ORDER BY cost_cad / refills DESC    -- expression in ORDER BY (alias also works)
"""
print(pd.read_sql(q3, conn).to_string(index=False))


---
## LIMIT and top-N queries


In [ ]:
# Top 3 most expensive prescriptions
print('=== Top 3 most expensive (LIMIT 3) ===')
q = """
SELECT rx_id, drug_name, cost_cad
FROM   prescriptions
ORDER BY cost_cad DESC
LIMIT 3
"""
print(pd.read_sql(q, conn).to_string(index=False))

# Top 1 per group — preview of window function approach
print()
print('=== Cheapest prescription per drug (GROUP BY approach) ===')
q2 = """
SELECT drug_name,
       MIN(cost_cad)  AS min_cost,
       MAX(cost_cad)  AS max_cost
FROM   prescriptions
GROUP BY drug_name
ORDER BY min_cost ASC
"""
print(pd.read_sql(q2, conn).to_string(index=False))

# LIMIT without ORDER BY — non-deterministic
print()
print('=== LIMIT without ORDER BY — row order is arbitrary ===')
print(pd.read_sql('SELECT rx_id, drug_name FROM prescriptions LIMIT 4', conn).to_string(index=False))
print('(The 4 rows returned may vary. Always pair LIMIT with ORDER BY.)')


---
## OFFSET and pagination


In [ ]:
# Classic OFFSET pagination
PAGE_SIZE = 4
print(f'=== Paginating {PAGE_SIZE} rows per page ===')
for page in range(4):
    offset = page * PAGE_SIZE
    q = f"""
SELECT rx_id, drug_name, dose_mg, cost_cad
FROM   prescriptions
ORDER BY rx_id
LIMIT  {PAGE_SIZE} OFFSET {offset}
"""
    result = pd.read_sql(q, conn)
    print(f'  Page {page+1} (OFFSET {offset}): {len(result)} rows')
    print(result.to_string(index=False))
    print()

print("""
Performance note:
  OFFSET n forces the database to read and discard n rows before returning results.
  LIMIT 10 OFFSET 10000 is much slower than LIMIT 10 OFFSET 0.
  For large tables, prefer keyset (cursor) pagination:

    -- Instead of OFFSET, track the last seen id:
    SELECT rx_id, drug_name, cost_cad
    FROM   prescriptions
    WHERE  rx_id > :last_seen_id    -- :last_seen_id = last rx_id from previous page
    ORDER BY rx_id
    LIMIT  4;
""")


---
## NULL sort order and custom sort sequences


In [ ]:
# NULLs sort last in SQLite ASC (same as PostgreSQL default)
print('=== patient_id ASC — NULLs sort last ===')
q = """
SELECT rx_id, patient_id, drug_name
FROM   prescriptions
ORDER BY patient_id ASC
"""
print(pd.read_sql(q, conn).to_string(index=False))

# Force NULLs first with CASE workaround (portable SQLite/PostgreSQL)
print()
print('=== Force NULLs first (portable CASE trick) ===')
q2 = """
SELECT rx_id, patient_id, drug_name
FROM   prescriptions
ORDER BY CASE WHEN patient_id IS NULL THEN 0 ELSE 1 END,
         patient_id ASC
"""
print(pd.read_sql(q2, conn).to_string(index=False))
print('PostgreSQL also supports: ORDER BY patient_id ASC NULLS FIRST')

# Custom business-defined sort order
print()
print('=== Custom drug priority ordering ===')
q3 = """
SELECT drug_name, cost_cad,
       CASE drug_name
           WHEN 'Metformin'    THEN 1
           WHEN 'Lisinopril'   THEN 2
           WHEN 'Atorvastatin' THEN 3
           WHEN 'Metoprolol'   THEN 4
           ELSE                    99
       END AS priority
FROM   prescriptions
ORDER BY priority, cost_cad DESC
"""
print(pd.read_sql(q3, conn).to_string(index=False))

conn.close()


---
## Common Pitfalls

**1. Assuming result order without ORDER BY**
SQL tables are unordered sets. Without ORDER BY, the database may return rows in any order and this can change between executions as data changes, indexes are rebuilt, or the query planner chooses a different plan. Always specify ORDER BY when order matters.

**2. LIMIT without ORDER BY gives non-deterministic results**
`SELECT * FROM prescriptions LIMIT 3` does not guarantee the same 3 rows across queries. It may return different rows after an INSERT or a VACUUM. Always pair LIMIT with ORDER BY.

**3. OFFSET pagination is O(n) — expensive at scale**
`LIMIT 10 OFFSET 50000` forces the database to internally read 50,010 rows and discard the first 50,000. At large offsets this becomes very slow. Use keyset (cursor) pagination (`WHERE id > :last_seen_id`) for production pagination on large tables.

**4. ORDER BY column position is fragile**
`ORDER BY 2, 3` sorts by the second and third columns of the SELECT list. If you add a column before them, the sort silently changes. Use explicit column names or aliases.

**5. NULL sort order differs across databases**
PostgreSQL and SQLite both sort NULLs last in ASC order by default, but MySQL sorts NULLs first in ASC. If you need portable code or predictable NULL placement, use the explicit `NULLS FIRST`/`NULLS LAST` clause (PostgreSQL) or the CASE workaround shown above.

**6. ORDER BY a non-indexed column on a large table is slow**
`ORDER BY last_name` on a million-row table without an index forces a full sort operation. Add an index on frequently sorted columns. In PostgreSQL, check with `EXPLAIN ANALYZE` whether the sort is using an index scan or a costly sequential sort.


---
*sql_methods_library - Samantha McGarrigle*